# 3. Searching the registry

Third notebook in the sequence. You've set up the database in [`1_Set_Up_Local_Database.ipynb`](./1_Set_Up_Local_Database.ipynb) and walked through making changes to individual records in [`2_Create_and_Update_Metadata.ipynb`](./2_Create_and_Update_Metadata.ipynb). 

Notebook 2 shows you how to register and update records: fetch a record by id, edit it, save it back. This notebook is about **finding** records — searching, filtering, and slicing across many assets at once. Cross-record questions like *"which projects produced the most assets in the past year?"* or *"give me all wt/wt subjects in this cohort with acquisitions between these dates."* Those involve filtering on fields that span multiple tables (subject genotype, acquisition time, project), and the client's list endpoints don't accept filter expressions.

For that kind of question, search against the **integrated document store** — a collection where each asset record carries its subject, instrument, acquisition, and quality control inline. One query can filter on any combination of those fields without needing joins.

## What this notebook covers

A tour of common search patterns:

- Look at one record so you know what fields are searchable.
- Pull a few core fields into a flat table, then filter and group with pandas.
- Ask *"which projects?"*, *"which modalities?"*, *"how many subjects per genotype?"*.
- Build a cohort: project + genotype + acquisition date range → list of asset locations.

**Prerequisites**: complete [`1_Set_Up_Local_Database.ipynb`](./1_Set_Up_Local_Database.ipynb) first to get the services running and the database populated.

## 1. Open a connection and verify the database

> **Note**: this notebook talks to MongoDB directly via `pymongo` for now. In the future this access will go through a higher-level Python API exposed by the `biodata_registry` package, so client code won't need to know about MongoDB specifically — the same queries will be expressible against the registry's API surface.

In [ ]:
from pymongo import MongoClient
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 80)

client = MongoClient(
    host="localhost",
    port=27017,
    username="admin",
    password="admin_password",
)
collection = client["metadata"]["data_assets"]
print(f"Connected. {collection.count_documents({})} documents in collection.")

## 2. See what one record looks like

Before slicing into fields, it helps to know what a single record contains. Each document is the merger of an asset and everything that joins to it: identifiers at the top, denormalized scalars next, the nested JSON payloads from each underlying row, and finally the embedded collections of subjects, procedures, and quality-control entries.

The cell below prints the top-level keys of one record along with what each field holds.

In [ ]:
sample = collection.find_one()

print(f"Top-level fields ({len(sample)} keys):\n")
for key, val in sample.items():
    if isinstance(val, dict):
        shape = f"dict, {len(val)} key(s)"
    elif isinstance(val, list):
        shape = f"list, {len(val)} item(s)"
    elif val is None:
        shape = "None"
    else:
        shape = f"{type(val).__name__} ({val!r:.60})"
    print(f"  {key:30s}  {shape}")

The fields fall into roughly four groups:

- **Identifiers**: `_id`, `data_asset_id`, `acquisition_id`, `instrument_id`, `process_id` — the row ids from each underlying table.
- **Top-level convenience fields**: `data_asset_name`, `data_asset_location`, `data_asset_external_links`, `instrument_name` — copied up so you can query or display them without reaching into the nested payloads.
- **Nested JSON payloads**: `data_asset_data`, `acquisition_data`, `instrument_data`, `processes_data` — the full content of each underlying row (this is where most of the schema-driven detail lives).
- **Embedded related records**: `subjects`, `subject_procedures`, `quality_control` — joined-in details so each asset record carries its subjects and QC alongside.

The searches that follow filter and aggregate on these fields directly.

## 3. Pull a few core fields into a flat table

Each record is richly nested. For most searches we only need a handful of fields per asset: project, subject, modality, data level, creation time, instrument, and location. Asking the database to return just those fields (instead of the whole record) keeps memory small and makes the downstream pandas code straightforward.

In [ ]:
projection = {
    "_id": 0,
    "data_asset_id": 1,
    "data_asset_name": 1,
    "location": "$data_asset_location",
    "project_name": "$data_asset_data.project_name",
    "subject_id": "$data_asset_data.subject_id",
    "data_level": "$data_asset_data.data_level",
    "creation_time": "$data_asset_data.creation_time",
    "modalities": "$data_asset_data.modalities.abbreviation",
    "instrument_name": 1,
    "institution": "$data_asset_data.institution.abbreviation",
}
df = pd.DataFrame(list(collection.find({}, projection)))
df["creation_time"] = pd.to_datetime(df["creation_time"], errors="coerce", utc=True)
df.head()

In [ ]:
summary = pd.Series({
    "total assets": len(df),
    "unique subjects": df["subject_id"].nunique(),
    "unique projects": df["project_name"].nunique(),
    "unique instruments": df["instrument_name"].nunique(),
    "date range": f"{df['creation_time'].min().date()} \u2192 {df['creation_time'].max().date()}",
})
summary.to_frame("value")

## 4. Which projects have produced the most assets?

Group by project name and count assets. This is the "where is the data coming from?" question — a useful starting point before drilling into any one project.

In [ ]:
by_project = (
    df.groupby("project_name", dropna=False)
      .agg(assets=("data_asset_id", "count"),
           subjects=("subject_id", "nunique"))
      .sort_values("assets", ascending=False)
)
by_project.head(10)

In [ ]:
top = by_project.head(10).iloc[::-1]
labels = [(p[:55] + "\u2026") if len(p) > 55 else p for p in top.index]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(labels, top["assets"], color="#4C72B0")
ax.set_xlabel("# data assets")
ax.set_title("Top 10 projects by asset count")
fig.tight_layout()
plt.show()

## 5. Broad-coverage vs. deep-coverage projects

Asset counts alone can be misleading. Two projects with 30 assets each can mean very different things: 30 different subjects with one asset apiece (broad survey) vs. 3 subjects with 10 assets apiece (longitudinal follow-up). The assets-per-subject ratio surfaces that distinction.

In [ ]:
by_project["assets_per_subject"] = (by_project["assets"] / by_project["subjects"]).round(2)
by_project.head(10)

In [ ]:
top = by_project.head(10).iloc[::-1]
labels = [(p[:55] + "\u2026") if len(p) > 55 else p for p in top.index]

fig, ax = plt.subplots(figsize=(9, 5))
y = range(len(top))
ax.barh([i - 0.2 for i in y], top["assets"], height=0.4, color="#4C72B0", label="assets")
ax.barh([i + 0.2 for i in y], top["subjects"], height=0.4, color="#DD8452", label="unique subjects")
ax.set_yticks(list(y))
ax.set_yticklabels(labels)
ax.set_xlabel("count")
ax.set_title("Assets vs. unique subjects (top 10 projects)")
ax.legend()
fig.tight_layout()
plt.show()

## 6. Count assets per modality

`modalities` is a list — one asset can be tagged with several (e.g. `behavior` together with `behavior-videos`). The cell below turns each asset into one row per modality before counting, so an asset tagged with two modalities contributes to both totals. The numbers will therefore add up to more than the total asset count.

In [ ]:
modality_counts = (
    df.explode("modalities")
      .groupby("modalities", dropna=False)
      .size()
      .sort_values(ascending=False)
      .rename("assets")
)
modality_counts.to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(modality_counts.index.astype(str), modality_counts.values, color="#55A868")
ax.set_ylabel("# data assets")
ax.set_title("Modality breakdown")
plt.xticks(rotation=30, ha="right")
fig.tight_layout()
plt.show()

## 7. Split between raw and derived assets

The `data_level` field tags each asset as either raw acquisition output or a derived/processed product. The ratio is a quick proxy for how much downstream analysis has happened on top of the raw collections.

In [ ]:
level_counts = df["data_level"].value_counts(dropna=False)
fig, ax = plt.subplots(figsize=(4, 4))
ax.pie(level_counts.values, labels=level_counts.index.astype(str),
       autopct="%1.0f%%", colors=["#4C72B0", "#DD8452", "#C44E52"])
ax.set_title("Data level distribution")
plt.show()
level_counts.to_frame("assets")

## 8. Acquisition cadence: assets created per month

Bucketing each asset's `creation_time` into monthly bins gives a view of throughput over time — when collection was ramping, when it plateaued, when it dropped off.

In [ ]:
monthly = (
    df.dropna(subset=["creation_time"])
      .set_index("creation_time")
      .resample("MS")
      .size()
)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly.index, monthly.values, marker="o", color="#4C72B0")
ax.set_ylabel("assets created")
ax.set_title("Assets created per month")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 9. Drill into the largest project: per-subject activity

Once you've spotted a project of interest, restrict the table to its assets and look at activity per subject — how many assets each subject contributes and the time window they span.

In [ ]:
target_project = by_project.index[0]
proj_df = df[df["project_name"] == target_project]
print(f"Project: {target_project}")
print(f"Assets: {len(proj_df)}  |  Unique subjects: {proj_df['subject_id'].nunique()}")

In [ ]:
subject_breakdown = (
    proj_df.groupby("subject_id")
           .agg(assets=("data_asset_id", "count"),
                first_seen=("creation_time", "min"),
                last_seen=("creation_time", "max"))
           .sort_values("assets", ascending=False)
)
subject_breakdown.head(10)

## 10. Compare modality coverage across the top projects

A project × modality table shows which projects are single-modality vs. multimodal, and where each modality concentrates. The heatmap that follows makes the dominant cells easy to spot at a glance.

In [ ]:
cross = (
    df.explode("modalities")
      .pivot_table(index="project_name", columns="modalities",
                   values="data_asset_id", aggfunc="count", fill_value=0)
      .loc[by_project.head(10).index]
)
cross

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(cross.values, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(cross.columns)))
ax.set_xticklabels(cross.columns, rotation=30, ha="right")
ax.set_yticks(range(len(cross.index)))
ax.set_yticklabels([(p[:50] + "\u2026") if len(p) > 50 else p for p in cross.index])
for i in range(cross.shape[0]):
    for j in range(cross.shape[1]):
        v = cross.values[i, j]
        if v:
            ax.text(j, i, str(v), ha="center", va="center",
                    color="white" if v > cross.values.max() / 2 else "black", fontsize=9)
ax.set_title("Project × modality (top 10 projects)")
fig.colorbar(im, ax=ax, label="# assets")
fig.tight_layout()
plt.show()

## 11. Reach into the embedded subject details: unique subjects per genotype

The flat table works well for the common cases, but the underlying records have a lot more inside them. Each asset record carries a `subjects` field — a mapping from subject id to that subject's full details (genotype, strain, date of birth, breeding info, and so on).

Below we ask the database to look inside that nested field and count distinct subjects per genotype. Because the subject details travel alongside each asset record, there's no separate lookup to do — the query stays a one-shot.

In [ ]:
pipeline = [
    {"$project": {
        "_id": 0,
        "subjects": {"$objectToArray": "$subjects"},
    }},
    {"$unwind": "$subjects"},
    {"$replaceRoot": {"newRoot": "$subjects.v"}},
    {"$group": {
        "_id": "$subject_details.genotype",
        "subjects": {"$addToSet": "$subject_id"},
    }},
    {"$project": {"genotype": "$_id", "_id": 0, "unique_subjects": {"$size": "$subjects"}}},
    {"$sort": {"unique_subjects": -1}},
]
pd.DataFrame(list(collection.aggregate(pipeline)))

## 12. Build a filtered cohort: asset locations for wt/wt subjects in one project, within a date range

A common workflow is to pull a focused slice of the registry: pick a project, restrict to a genotype, and narrow to a window of dates. The result is a small DataFrame of asset locations you can hand off to downstream analysis.

We pull the project's records along with each one's embedded subject details, then apply the genotype and date filters in pandas. The project filter is the only one pushed to the database; the remaining work is in-memory and stays readable.

In [ ]:
project = "Cognitive flexibility in patch foraging"
start = pd.Timestamp("2025-01-01", tz="UTC")
end = pd.Timestamp("2025-09-30", tz="UTC")
target_genotype = "wt/wt"

docs = list(collection.find(
    {"data_asset_data.project_name": project},
    {
        "_id": 0,
        "data_asset_id": 1,
        "data_asset_name": 1,
        "data_asset_location": 1,
        "data_asset_data.subject_id": 1,
        "acquisition_data.acquisition_start_time": 1,
        "subjects": 1,
    },
))

# Each asset record has a `subjects` map keyed by an internal id. Pull out the genotype
# for each subject attached to the asset and flag the asset as a match if any subject
# matches the target genotype.
rows = []
for d in docs:
    dd = d.get("data_asset_data", {})
    genotypes = [
        (s.get("subject_details") or {}).get("genotype", "").strip()
        for s in (d.get("subjects") or {}).values()
    ]
    rows.append({
        "data_asset_id": d["data_asset_id"],
        "name": d.get("data_asset_name"),
        "subject_id": dd.get("subject_id"),
        "acquisition_start": (d.get("acquisition_data") or {}).get("acquisition_start_time"),
        "location": d.get("data_asset_location"),
        "is_match": any(g == target_genotype for g in genotypes),
    })

cohort = pd.DataFrame(rows)
cohort["acquisition_start"] = pd.to_datetime(cohort["acquisition_start"], errors="coerce", utc=True)

filtered = (
    cohort[cohort["is_match"]
           & cohort["acquisition_start"].between(start, end)]
    .drop(columns="is_match")
    .sort_values("acquisition_start")
    .reset_index(drop=True)
)

print(f"{len(filtered)} {target_genotype} assets in '{project}' between {start.date()} and {end.date()}")
filtered